In [0]:
from pyspark.sql import functions as F

SILVER     = "nyc_taxi.silver.yellow_trips"
GOLD_DAILY = "nyc_taxi.gold.trips_daily"

START_VERSION = 8   # 水位线：Gold 上次全量构建之后的第一个数据变更版本
END_VERSION   = spark.sql(f"DESCRIBE HISTORY {SILVER} LIMIT 1").first()["version"]
print(f"处理 Silver 版本 {START_VERSION} → {END_VERSION}")

def daily_agg(df):
    # ⚠️ 把这里的聚合换成 04_gold 里 trips_daily 的原样代码（包括 round 之类的处理）
    return (df.groupBy("pickup_date").agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("total_revenue"),
        F.sum("passenger_count").alias("total_passengers"),
        F.round(F.avg("trip_duration_min"), 2).alias("avg_duration_min"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
    ))

In [0]:
changes = spark.sql(f"SELECT * FROM table_changes('{SILVER}', {START_VERSION}, {END_VERSION})")
display(changes.groupBy("_commit_version", "_change_type").count().orderBy("_commit_version", "_change_type"))

affected = [r.pickup_date for r in
            changes.select("pickup_date").distinct().orderBy("pickup_date").collect()]
print("受影响日期:", affected)

In [0]:
gold_ver_before = spark.sql(f"DESCRIBE HISTORY {GOLD_DAILY} LIMIT 1").first()["version"]

pred = "pickup_date IN (" + ", ".join(f"DATE'{d}'" for d in affected) + ")"
print("replaceWhere:", pred)

new_rows = daily_agg(spark.table(SILVER).filter(F.col("pickup_date").isin(affected)))

(new_rows.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", pred)
    .saveAsTable(GOLD_DAILY))

In [0]:
print(gold_ver_before)

In [0]:
display(spark.sql(f"""
SELECT 'before' AS v, * FROM {GOLD_DAILY} VERSION AS OF {gold_ver_before} WHERE {pred}
UNION ALL
SELECT 'after'  AS v, * FROM {GOLD_DAILY} WHERE {pred}
ORDER BY pickup_date, v DESC
"""))

In [0]:
full = daily_agg(spark.table(SILVER)).alias("f")
gold = spark.table(GOLD_DAILY).alias("g")
chk  = full.join(gold, "pickup_date", "full_outer")

def mismatch(c):   # 整数列：要求完全相等（null 安全）
    return F.sum((~F.col(f"f.{c}").eqNullSafe(F.col(f"g.{c}"))).cast("int")).alias(f"{c}_mismatch_days")

def maxdiff(c):    # 浮点列：看最大差值
    return F.max(F.abs(F.col(f"f.{c}") - F.col(f"g.{c}"))).alias(f"{c}_max_diff")

display(chk.agg(
    F.count("*").alias("days"),
    mismatch("trip_count"), mismatch("total_passengers"),
    maxdiff("total_revenue"), maxdiff("avg_duration_min"), maxdiff("avg_distance_miles"),
))
print("Gold Σ trip_count:", gold.agg(F.sum("trip_count")).first()[0], "（应为 21,965,805）")